# Neural Network Baseline

In this section, we train a baseline neural network on the text data, showcasing initial results from the evaluation plan at the end. Many of the findings from the baseline model will determine future approaches that are taken to improve the model. 

As a first step, the data is loaded, duplicates are dropped, and the labels are extracted.

In [30]:
import pandas as pd
from pathlib import Path

data_path = Path("data/clean_data.csv")

df = pd.read_csv(data_path)

df = df.drop_duplicates(subset=['Description', 'Type']) # Drop duplicates between text and label
df = df.drop_duplicates(subset=['Description'], keep=False) # Drop duplicate text with different labels

texts = df["Description"].values
labels = df["Type"].values

print(df["Type"].value_counts())

Type
Grievance             6482
Required Support      2192
Enquiry               1186
Complaint              547
Suggestion              82
Willing to Support      20
Appreciation            10
Name: count, dtype: int64


Some classes are rarer than others, which needs to be accounted for in the train/test split.

# 1. Encode Target Labels

All of the target label emotions need to be encoded as integers in order to be predicted by the neural network.

In [31]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(labels)

# Print the new encoded labels
print(y)

[2 3 4 ... 1 5 3]


# 2. Data Splits

The data is split according to the evaluation plan. First, a 85/15 train/test split and then, a train/validation split. The split of the data is stratified to try to keep the proportions the same across datasets for the rarer classes.

In [32]:
from sklearn.model_selection import train_test_split

# Train/Test split
x_train, x_test, y_train, y_test = train_test_split(
    texts, y,
    test_size=0.15,
    stratify=y,
    random_state=42
)

# Train/Validation split
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=42
)

print(f"Number of observations in training set: " + str(len(x_train)))
print(f"Number of observations in validation set: " + str(len(x_val)))
print(f"Number of observations in test set: " + str(len(x_test)))

Number of observations in training set: 7599
Number of observations in validation set: 1342
Number of observations in test set: 1578


# 3. Building The Vocabulary

A CNN needs each observation to be the same length so the training set forms a rectangle. In order to decide the length, we need to find the max length of the observations first, decide on a sequence length, and then pad the rest of the observations.

In [33]:
import numpy as np

lengths = [len(t.split()) for t in x_train]

print("Mean observation length: " + str(np.mean(lengths)))
print("Longest observation length: " + str(np.max(lengths)))

Mean observation length: 61.128437952362155
Longest observation length: 182


For the baseline model, we will use 182 as the sequence length since it is the longest observation in the training set. However, we will explore capping the sequence length in the future. Now, we need to count the tokens in the training set to see if we need to cap the number.

In [ ]:
from collections import Counter
import re

def tokenize(text): # Tokenize based on whitespace
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.split()

token_counts = Counter()
for text in x_train:
    token_counts.update(tokenize(text))

train_vocab = ["<pad>", "<unk>"] + [word for word, count in token_counts.most_common()]

word2idx = {word: idx for idx, word in enumerate(train_vocab)}

print("Full vocabulary size:", len(train_vocab))

Full vocabulary size: 14449


The total vocabulary size of the training set isn't very big so we'll keep the full vocabulary. However, we may explore capping the maximum tokens later on.

In [ ]:
SEQ_LEN = 182 # Sequence length: set with the longest observation
PAD = 0
UNK = 1

def encode(text):
    ids = [word2idx.get(word, UNK) for word in tokenize(text)]
    ids = ids[:SEQ_LEN] # truncate if too long
    ids = ids + [PAD] * (SEQ_LEN - len(ids)) # pad if too short
    return ids

Now that the vocabulary has been built from the training set, we can apply the same vectorizer to all three of the data sets.

In [ ]:
import torch

# Apply the vectorizer to the three data sets
x_train_vec = torch.tensor([encode(t) for t in x_train], dtype=torch.long)
x_val_vec = torch.tensor([encode(t) for t in x_val], dtype=torch.long)
x_test_vec = torch.tensor([encode(t) for t in x_test], dtype=torch.long)

print(x_train_vec.shape) # Make sure that the new training set shape is correct

torch.Size([7599, 182])


# 4. Build The CNN Model

Now we will construct the baseline structure of the Convolutional Neural Network that will be used to take in the text to predict the sentiment label. The structure is defined first including key hyperparameters before any training can occur. 

In [ ]:
import torch.nn as nn

VOCAB_SIZE = len(train_vocab)
EMBED_DIM = 64 # embedding dimensions
NUM_CLASSES = 7

class BaselineCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv = nn.Conv1d(in_channels=embed_dim, out_channels=64, kernel_size=5) # keep kernel size small
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.embedding(x)        # (batch, SEQ_LEN, 64)
        x = x.permute(0, 2, 1)       # (batch, 64, SEQ_LEN)
        x = torch.relu(self.conv(x)) # (batch, 64, SEQ_LEN - 4)
        x = x.max(dim=2).values      # (batch, 64)  global max pooling
        return self.fc(x)            # (batch, 7)   raw scores, no softmax

model = BaselineCNN(VOCAB_SIZE, EMBED_DIM, NUM_CLASSES)
print(model)

BaselineCNN(
  (embedding): Embedding(14449, 64, padding_idx=0)
  (conv): Conv1d(64, 64, kernel_size=(5,), stride=(1,))
  (fc): Linear(in_features=64, out_features=7, bias=True)
)


The model is set up to be a baseline with the following key hyperparameters:
* embed_dim: 64. This is a good starting place as it has fewer parameters and the dataset is smaller.
* out_channels: 64. This is a good starting place for one kernel size and a smaller dataset.
* kernel_size: 5. The model will look at five words at a time, which is a good starting place to cover shorter descriptions.
* activation: ReLU. This is the most commonly used activation function in deep learning.
* layers: one convolution layer and one linear output layer. This is a simple setup that we will later look to enhance with dropout.

# 5. Training The Model

Since the model structure is defined, we are ready to start the training loop. The evaluation function is defined first at the top as well as the loss. Then, the training loop is shown over 50 epochs (or until early stopping occurs) with validation metrics.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

criterion = nn.CrossEntropyLoss() # loss metric

def evaluate(model, loader):
    model.eval() # needed during eval stage
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * len(yb)
            all_preds.extend(logits.argmax(dim=1).tolist())
            all_labels.extend(yb.tolist())

    return {
        "loss": total_loss / len(all_labels),
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1": f1_score(all_labels, all_preds, average="macro"),
        "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall": recall_score(all_labels, all_preds, average="macro", zero_division=0),
    }

Loss: nn.CrossEntropyLoss() was picked because this is multi-class classification with integers numbered 0-6.

In [39]:
from torch.utils.data import TensorDataset, DataLoader

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t   = torch.tensor(y_val, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(x_train_vec, y_train_t)
val_ds = TensorDataset(x_val_vec, y_val_t)
test_ds = TensorDataset(x_test_vec, y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

Optimizer: "adam" was picked instead of "adamw" for the baseline model. Later on, we'll switch to adamw to add weight decay as a comparison to this model.

In [41]:
torch.manual_seed(42)   # reproducible weight initialization and shuffling

model = BaselineCNN(VOCAB_SIZE, EMBED_DIM, NUM_CLASSES) #initialize weights
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) # model optimizer

EPOCHS = 50 # set max epochs
PATIENCE = 15

hist = []
best_loss, best_epoch, best_weights, waited = float("inf"), -1, None, 0

for epoch in range(1, EPOCHS+1):
    # ---- train ----
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(yb)

    train_loss /= len(train_ds)

    # ---- validate ----
    val_metrics = evaluate(model, val_loader)
    hist.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}})

    print(f"Epoch {epoch:2d} | train_loss {train_loss:.4f} | "
          f"val_loss {val_metrics['loss']:.4f} | val_acc {val_metrics['accuracy']:.4f} | "
          f"val_f1 {val_metrics['f1']:.4f}")

    # ---- early stopping ----
    if val_metrics["loss"] < best_loss:
        best_loss = val_metrics["loss"]
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        waited = 0
        best_epoch = epoch
    else:
        waited += 1
        if waited >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_weights)   # restore the best weights

print(f"Ran {len(hist)} epochs, kept epoch {best_epoch}")
print(f"Validation loss at the epoch we kept : {best_loss}")

Epoch  1 | train_loss 1.1144 | val_loss 1.0864 | val_acc 0.6073 | val_f1 0.1461
Epoch  2 | train_loss 0.9659 | val_loss 1.0507 | val_acc 0.6006 | val_f1 0.1610
Epoch  3 | train_loss 0.8638 | val_loss 1.0648 | val_acc 0.5961 | val_f1 0.1520
Epoch  4 | train_loss 0.7497 | val_loss 1.0701 | val_acc 0.6088 | val_f1 0.1612
Epoch  5 | train_loss 0.6124 | val_loss 1.0959 | val_acc 0.5745 | val_f1 0.1849
Epoch  6 | train_loss 0.4709 | val_loss 1.1512 | val_acc 0.5879 | val_f1 0.1872
Epoch  7 | train_loss 0.3479 | val_loss 1.2091 | val_acc 0.5648 | val_f1 0.1977
Epoch  8 | train_loss 0.2463 | val_loss 1.2974 | val_acc 0.5484 | val_f1 0.1910
Epoch  9 | train_loss 0.1705 | val_loss 1.3537 | val_acc 0.5760 | val_f1 0.1899
Epoch 10 | train_loss 0.1197 | val_loss 1.4382 | val_acc 0.5402 | val_f1 0.1942
Epoch 11 | train_loss 0.0832 | val_loss 1.4934 | val_acc 0.5745 | val_f1 0.1945
Epoch 12 | train_loss 0.0635 | val_loss 1.5788 | val_acc 0.5678 | val_f1 0.1866
Epoch 13 | train_loss 0.0492 | val_loss 

The baseline model overfits immediately since validation loss is minimized at epoch 2 and increases after while training loss goes down. At epoch 2, validation accuracy is at 60.06%, slightly below the majority class baseline of 61.6%. The macro F1 does indicate, however, that the model does capture some other classes beyond the majority class.

# 6. Test Set Results

# 7. Evaluation

# 8. Conclusion